
# Noise Sensitivity Analysis for Fisher-Regularized Neural Surrogates

This notebook runs the noise sensitivity experiment for the Fisher-regularized Gaussian variance dynamics:

\[
\dot{u} = 2(1-u) + \frac{\varepsilon}{u}.
\]

It compares:

1. **Direct MLP**: learns \( (u_0,\varepsilon,t) \mapsto u(t;u_0,\varepsilon) \).
2. **Residual MLP**: learns the Fisher-induced correction relative to the unregularized Ornstein--Uhlenbeck baseline:

\[
\widehat{u}(t;u_0,\varepsilon)
=
u_{\mathrm{OU}}(t;u_0)
+
\varepsilon r_\theta(u_0,\varepsilon,t).
\]

Noise is added only to the **training targets**, while validation and test trajectories remain clean.
The notebook reports \(E_{\mathrm{traj}}\), \(E_{\mathrm{cross}}\), \(E_{\mathrm{over}}\), and \(E_{\mathrm{eq}}\), summarized as mean \(\pm\) 95% confidence-interval half-width over \(S=10\) random seeds.


In [ ]:

import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

try:
    from scipy.stats import t as student_t
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.11.0+cpu
CUDA available: False


## Configuration

In [ ]:

CONFIG = {
    "n_pairs": 300,
    "n_time": 80,
    "t_max": 5.0,

    "u0_min": 0.2,
    "u0_max": 2.5,
    "eps_min": 0.01,
    "eps_max": 0.5,

    "n_seeds": 10,
    "data_seed": 123,
    "split_seed": 999,
    "base_seed": 2026,

    "epochs": 1500,
    "batch_size": 512,
    "lr": 2e-3,
    "patience": 150,

    # Set True for a quick notebook test.
    "quick_test": False,
}

if CONFIG["quick_test"]:
    CONFIG.update({
        "n_pairs": 80,
        "n_time": 50,
        "n_seeds": 2,
        "epochs": 200,
        "patience": 40,
    })

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


'cpu'

## Fisher-Regularized Variance Dynamics

In [ ]:

def u_star(eps):
    """Fisher-regularized equilibrium variance."""
    eps = np.asarray(eps)
    return (1.0 + np.sqrt(1.0 + 2.0 * eps)) / 2.0


def u_ou(t, u0):
    """Unregularized Ornstein--Uhlenbeck variance trajectory."""
    return 1.0 + (u0 - 1.0) * np.exp(-2.0 * t)


def rhs(u, eps):
    """Right-hand side: du/dt = 2(1-u) + eps/u."""
    return 2.0 * (1.0 - u) + eps / np.maximum(u, 1e-12)


def integrate_rk4(u0, eps, t_grid):
    """Fourth-order Runge--Kutta integration on a prescribed time grid."""
    u = np.zeros_like(t_grid, dtype=np.float64)
    u[0] = float(u0)

    for k in range(1, len(t_grid)):
        dt = t_grid[k] - t_grid[k - 1]
        uk = u[k - 1]

        k1 = rhs(uk, eps)
        k2 = rhs(uk + 0.5 * dt * k1, eps)
        k3 = rhs(uk + 0.5 * dt * k2, eps)
        k4 = rhs(uk + dt * k3, eps)

        u[k] = uk + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        u[k] = max(u[k], 1e-12)

    return u


## Dataset Generation and Fixed Parameter-Pair Split

In [ ]:

def make_dataset(
    n_pairs,
    n_time,
    u0_range=(0.2, 2.5),
    eps_range=(0.01, 0.5),
    t_max=5.0,
    seed=123,
):
    """Generate clean trajectories and return a long-form dataframe."""
    rng = np.random.default_rng(seed)

    u0s = rng.uniform(u0_range[0], u0_range[1], size=n_pairs)
    epss = rng.uniform(eps_range[0], eps_range[1], size=n_pairs)
    t_grid = np.linspace(0.0, t_max, n_time)

    rows = []
    for pair_id, (u0_i, eps_i) in enumerate(zip(u0s, epss)):
        u_clean = integrate_rk4(u0_i, eps_i, t_grid)

        for t_i, u_i in zip(t_grid, u_clean):
            rows.append({
                "pair_id": pair_id,
                "u0": u0_i,
                "eps": eps_i,
                "t": t_i,
                "u_clean": u_i,
            })

    return pd.DataFrame(rows)


def split_pair_ids(n_pairs, train_frac=0.70, val_frac=0.15, seed=999):
    """Split by trajectory/parameter-pair IDs to avoid trajectory leakage."""
    rng = np.random.default_rng(seed)
    ids = np.arange(n_pairs)
    rng.shuffle(ids)

    n_train = int(train_frac * n_pairs)
    n_val = int(val_frac * n_pairs)

    train_ids = set(ids[:n_train])
    val_ids = set(ids[n_train:n_train+n_val])
    test_ids = set(ids[n_train+n_val:])

    return train_ids, val_ids, test_ids


df = make_dataset(
    n_pairs=CONFIG["n_pairs"],
    n_time=CONFIG["n_time"],
    u0_range=(CONFIG["u0_min"], CONFIG["u0_max"]),
    eps_range=(CONFIG["eps_min"], CONFIG["eps_max"]),
    t_max=CONFIG["t_max"],
    seed=CONFIG["data_seed"],
)

train_ids, val_ids, test_ids = split_pair_ids(CONFIG["n_pairs"], seed=CONFIG["split_seed"])

train_clean = df[df["pair_id"].isin(train_ids)].copy()
val_clean = df[df["pair_id"].isin(val_ids)].copy()
test_clean = df[df["pair_id"].isin(test_ids)].copy()

print(df.head())
print("Train/Val/Test shapes:", train_clean.shape, val_clean.shape, test_clean.shape)


   pair_id        u0       eps         t   u_clean
0        0  1.769409  0.145374  0.000000  1.769409
1        0  1.769409  0.145374  0.063291  1.682941
2        0  1.769409  0.145374  0.126582  1.607001
3        0  1.769409  0.145374  0.189873  1.540327
4        0  1.769409  0.145374  0.253165  1.481808
Train/Val/Test shapes: (16800, 5) (3600, 5) (3600, 5)


## Model Definitions

In [ ]:

class MLP(nn.Module):
    def __init__(self, input_dim=3, hidden=(64, 64), activation="tanh"):
        super().__init__()

        if activation == "relu":
            act = nn.ReLU
        elif activation == "gelu":
            act = nn.GELU
        else:
            act = nn.Tanh

        layers = []
        prev = input_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(act())
            prev = h
        layers.append(nn.Linear(prev, 1))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


@dataclass
class Scaler:
    mean: np.ndarray
    std: np.ndarray

    def transform(self, x):
        return (x - self.mean) / self.std

    def inverse(self, x):
        return x * self.std + self.mean


def fit_scaler(x):
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-12, 1.0, std)
    return Scaler(mean=mean, std=std)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## Training and Prediction Utilities

In [ ]:

def train_model(
    x_train,
    y_train,
    x_val,
    y_val,
    seed,
    epochs=1500,
    batch_size=512,
    lr=2e-3,
    patience=150,
    device="cpu",
):
    set_seed(seed)

    x_scaler = fit_scaler(x_train)
    y_scaler = fit_scaler(y_train)

    x_train_s = x_scaler.transform(x_train).astype(np.float32)
    y_train_s = y_scaler.transform(y_train).astype(np.float32)
    x_val_s = x_scaler.transform(x_val).astype(np.float32)
    y_val_s = y_scaler.transform(y_val).astype(np.float32)

    train_ds = TensorDataset(torch.tensor(x_train_s), torch.tensor(y_train_s))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = MLP(input_dim=x_train.shape[1], hidden=(64, 64), activation="tanh").to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    x_val_t = torch.tensor(x_val_s, dtype=torch.float32, device=device)
    y_val_t = torch.tensor(y_val_s, dtype=torch.float32, device=device)

    best_state = None
    best_val = float("inf")
    wait = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(x_val_t), y_val_t).item()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, x_scaler, y_scaler


def predict_model(model, x, x_scaler, y_scaler, device="cpu"):
    model.eval()
    x_s = x_scaler.transform(x).astype(np.float32)

    with torch.no_grad():
        pred_s = model(torch.tensor(x_s, dtype=torch.float32, device=device)).cpu().numpy()

    return y_scaler.inverse(pred_s)


## Mechanism-Level Metrics

In [ ]:

def crossing_time(t, u, level=1.0):
    """First crossing time from below. Returns None if no crossing."""
    if u[0] >= level:
        return None

    above = np.where(u >= level)[0]
    if len(above) == 0:
        return None

    k = above[0]
    if k == 0:
        return t[0]

    t0, t1 = t[k-1], t[k]
    u0, u1 = u[k-1], u[k]

    if abs(u1 - u0) < 1e-12:
        return t1

    return t0 + (level - u0) * (t1 - t0) / (u1 - u0)


def compute_metrics(test_pred_df):
    """Compute aggregate metrics over clean test trajectories."""
    e_traj_values = []
    e_cross_values = []
    e_over_values = []
    e_eq_values = []

    for pair_id, g in test_pred_df.groupby("pair_id"):
        g = g.sort_values("t")

        t = g["t"].to_numpy()
        u_true = g["u_clean"].to_numpy()
        u_pred = g["u_pred"].to_numpy()
        eps = float(g["eps"].iloc[0])
        ueq = float(u_star(np.array([eps]))[0])

        e_traj_values.append(np.mean((u_pred - u_true) ** 2))

        tc_true = crossing_time(t, u_true, level=1.0)
        if tc_true is not None:
            tc_pred = crossing_time(t, u_pred, level=1.0)
            if tc_pred is None:
                tc_pred = t[-1]
            e_cross_values.append(abs(tc_pred - tc_true))

        a_true = max(0.0, float(np.max(u_true) - ueq))
        a_pred = max(0.0, float(np.max(u_pred) - ueq))
        e_over_values.append(abs(a_pred - a_true))

        e_eq_values.append(abs(float(u_pred[-1]) - ueq))

    return {
        "E_traj": float(np.mean(e_traj_values)),
        "E_cross": float(np.mean(e_cross_values)) if len(e_cross_values) else np.nan,
        "E_over": float(np.mean(e_over_values)),
        "E_eq": float(np.mean(e_eq_values)),
    }


## Run Noise Sensitivity Experiment

In [ ]:

def run_noise_sensitivity(config=CONFIG, device=DEVICE):
    clean_train_std = train_clean["u_clean"].std()

    noise_settings = [
        ("0%", 0.0),
        ("0.1%", 0.001),
        ("1%", 0.01),
        ("5%", 0.05),
    ]

    all_rows = []

    for noise_label, r_noise in noise_settings:
        print(f"\nNoise level: {noise_label}")
        sigma_n = r_noise * clean_train_std

        for seed_idx in range(config["n_seeds"]):
            run_seed = config["base_seed"] + seed_idx
            rng = np.random.default_rng(run_seed + 10000)

            train_df = train_clean.copy()
            val_df = val_clean.copy()
            test_df = test_clean.copy()

            # Noise is added only to training targets.
            train_df["u_target"] = train_df["u_clean"] + rng.normal(
                loc=0.0,
                scale=sigma_n,
                size=len(train_df),
            )

            # Validation and test remain clean.
            val_df["u_target"] = val_df["u_clean"]
            test_df["u_target"] = test_df["u_clean"]

            x_train = train_df[["u0", "eps", "t"]].to_numpy(dtype=np.float64)
            x_val = val_df[["u0", "eps", "t"]].to_numpy(dtype=np.float64)
            x_test = test_df[["u0", "eps", "t"]].to_numpy(dtype=np.float64)

            # Direct MLP
            y_train_direct = train_df[["u_target"]].to_numpy(dtype=np.float64)
            y_val_direct = val_df[["u_target"]].to_numpy(dtype=np.float64)

            direct_model, direct_xs, direct_ys = train_model(
                x_train=x_train,
                y_train=y_train_direct,
                x_val=x_val,
                y_val=y_val_direct,
                seed=run_seed,
                epochs=config["epochs"],
                batch_size=config["batch_size"],
                lr=config["lr"],
                patience=config["patience"],
                device=device,
            )

            direct_pred = predict_model(
                direct_model,
                x_test,
                direct_xs,
                direct_ys,
                device=device,
            ).reshape(-1)

            pred_df = test_df.copy()
            pred_df["u_pred"] = direct_pred
            direct_metrics = compute_metrics(pred_df)

            all_rows.append({
                "noise_level": noise_label,
                "model": "Direct MLP",
                "seed": seed_idx,
                **direct_metrics,
            })

            # Residual MLP
            train_ou = u_ou(train_df["t"].to_numpy(), train_df["u0"].to_numpy())
            val_ou = u_ou(val_df["t"].to_numpy(), val_df["u0"].to_numpy())
            test_ou = u_ou(test_df["t"].to_numpy(), test_df["u0"].to_numpy())

            train_eps = train_df["eps"].to_numpy()
            val_eps = val_df["eps"].to_numpy()
            test_eps = test_df["eps"].to_numpy()

            y_train_res = ((train_df["u_target"].to_numpy() - train_ou) / train_eps).reshape(-1, 1)
            y_val_res = ((val_df["u_target"].to_numpy() - val_ou) / val_eps).reshape(-1, 1)

            residual_model, residual_xs, residual_ys = train_model(
                x_train=x_train,
                y_train=y_train_res,
                x_val=x_val,
                y_val=y_val_res,
                seed=run_seed + 5000,
                epochs=config["epochs"],
                batch_size=config["batch_size"],
                lr=config["lr"],
                patience=config["patience"],
                device=device,
            )

            residual_r_pred = predict_model(
                residual_model,
                x_test,
                residual_xs,
                residual_ys,
                device=device,
            ).reshape(-1)

            residual_pred = test_ou + test_eps * residual_r_pred

            pred_df = test_df.copy()
            pred_df["u_pred"] = residual_pred
            residual_metrics = compute_metrics(pred_df)

            all_rows.append({
                "noise_level": noise_label,
                "model": "Residual MLP",
                "seed": seed_idx,
                **residual_metrics,
            })

            print(
                f"  seed {seed_idx:02d}: "
                f"Direct E_traj={direct_metrics['E_traj']:.2e}, "
                f"Residual E_traj={residual_metrics['E_traj']:.2e}"
            )

    return pd.DataFrame(all_rows)


# This cell may take several minutes for the full paper-quality setting.
seed_results = run_noise_sensitivity()
seed_results.head()



Noise level: 0%
  seed 00: Direct E_traj=1.04e-07, Residual E_traj=4.05e-09
  seed 01: Direct E_traj=1.04e-07, Residual E_traj=3.24e-09
  seed 02: Direct E_traj=1.06e-07, Residual E_traj=5.60e-09
  seed 03: Direct E_traj=1.43e-07, Residual E_traj=7.05e-09
  seed 04: Direct E_traj=1.16e-07, Residual E_traj=5.08e-09
  seed 05: Direct E_traj=2.28e-07, Residual E_traj=4.02e-09
  seed 06: Direct E_traj=1.38e-07, Residual E_traj=3.19e-09
  seed 07: Direct E_traj=1.29e-07, Residual E_traj=5.47e-09
  seed 08: Direct E_traj=6.84e-08, Residual E_traj=5.17e-09
  seed 09: Direct E_traj=1.35e-07, Residual E_traj=3.73e-09

Noise level: 0.1%
  seed 00: Direct E_traj=1.51e-07, Residual E_traj=2.40e-08
  seed 01: Direct E_traj=1.86e-07, Residual E_traj=1.42e-08
  seed 02: Direct E_traj=1.04e-07, Residual E_traj=2.09e-08
  seed 03: Direct E_traj=1.09e-07, Residual E_traj=2.54e-08
  seed 04: Direct E_traj=9.71e-08, Residual E_traj=1.45e-08
  seed 05: Direct E_traj=1.01e-07, Residual E_traj=2.10e-08
  se

## Summarize Across Seeds

In [ ]:

def summarize_seed_results(seed_df, n_seeds):
    """Mean and 95% CI half-width across seeds."""
    if SCIPY_AVAILABLE:
        tcrit = float(student_t.ppf(0.975, n_seeds - 1))
    else:
        # t_{0.975,9} for S=10.
        tcrit = 2.262

    rows = []
    for (noise_level, model), g in seed_df.groupby(["noise_level", "model"]):
        row = {
            "noise_level": noise_level,
            "model": model,
        }

        for metric in ["E_traj", "E_cross", "E_over", "E_eq"]:
            vals = g[metric].to_numpy(dtype=float)
            mean = np.nanmean(vals)
            std = np.nanstd(vals, ddof=1)
            n = np.sum(~np.isnan(vals))
            ci = tcrit * std / math.sqrt(n) if n > 1 else np.nan

            row[f"{metric}_mean"] = mean
            row[f"{metric}_ci"] = ci

        rows.append(row)

    return pd.DataFrame(rows)


summary = summarize_seed_results(seed_results, CONFIG["n_seeds"])
summary


## Generate LaTeX Table

In [ ]:

def fmt_metric(mean, ci):
    if np.isnan(mean):
        return "N/A"
    if np.isnan(ci):
        return f"{mean:.3e}"
    return f"{mean:.3e} $\\pm$ {ci:.1e}"


def make_latex_table(summary_df):
    order_noise = ["0%", "0.1%", "1%", "5%"]
    order_model = ["Direct MLP", "Residual MLP"]

    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(
        r"\caption{Noise sensitivity of neural surrogate models. Gaussian noise is added only to the training targets, while validation and test trajectories remain noise-free. Results are reported as mean $\pm$ 95\% confidence-interval half-width over $S=10$ random seeds. Lower values indicate better performance.}"
    )
    lines.append(r"\label{tab:noise_sensitivity}")
    lines.append(r"\begin{tabular}{llcccc}")
    lines.append(r"\hline")
    lines.append(
        r"Noise level & Model & $E_{\mathrm{traj}}$ & $E_{\mathrm{cross}}$ & $E_{\mathrm{over}}$ & $E_{\mathrm{eq}}$ \\"
    )
    lines.append(r"\hline")

    for nl in order_noise:
        for model in order_model:
            g = summary_df[
                (summary_df["noise_level"] == nl)
                & (summary_df["model"] == model)
            ]

            if len(g) == 0:
                continue

            row = g.iloc[0]
            vals = [
                fmt_metric(row["E_traj_mean"], row["E_traj_ci"]),
                fmt_metric(row["E_cross_mean"], row["E_cross_ci"]),
                fmt_metric(row["E_over_mean"], row["E_over_ci"]),
                fmt_metric(row["E_eq_mean"], row["E_eq_ci"]),
            ]

            lines.append(
                f"${nl}$ & {model} & {vals[0]} & {vals[1]} & {vals[2]} & {vals[3]} \\\\"
            )

    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    return "\n".join(lines)


latex_table = make_latex_table(summary)
print(latex_table)


## Save Results

In [ ]:

seed_results.to_csv("noise_sensitivity_seed_results.csv", index=False)
summary.to_csv("noise_sensitivity_summary.csv", index=False)

with open("noise_sensitivity_latex_table.tex", "w", encoding="utf-8") as f:
    f.write(latex_table)

print("Saved:")
print("  noise_sensitivity_seed_results.csv")
print("  noise_sensitivity_summary.csv")
print("  noise_sensitivity_latex_table.tex")


## Optional Visualization

In [ ]:

plot_df = summary.copy()

noise_order = ["0%", "0.1%", "1%", "5%"]
x = np.arange(len(noise_order))

plt.figure(figsize=(7, 4))

for model in ["Direct MLP", "Residual MLP"]:
    g = plot_df[plot_df["model"] == model].set_index("noise_level").loc[noise_order]
    y = g["E_traj_mean"].to_numpy()
    yerr = g["E_traj_ci"].to_numpy()
    plt.errorbar(x, y, yerr=yerr, marker="o", capsize=3, label=model)

plt.xticks(x, noise_order)
plt.yscale("log")
plt.xlabel("Training-target noise level")
plt.ylabel(r"$E_{\mathrm{traj}}$")
plt.title("Noise sensitivity: trajectory error")
plt.legend()
plt.tight_layout()
plt.show()
